# FIN 450 Case 4 — Apple Module 5 Greeks Extension

**Valuation date:** 2026-11-09. This extension preserves the original classroom notebook and its CRR function/tests. Real AAPL quotes and analyst targets are sourced, but 4% risk-free rate, expiration-level IV, 12-month analyst horizon and old DCF date are explicitly flagged. American CRR pricing is compared with a **European Black–Scholes benchmark** for analytic Greeks. This is not a claim that European Greeks equal American-option Greeks.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, math, json
from scipy.stats import norm
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
M=pd.read_csv(DATA/'company_inputs.csv').iloc[0].to_dict(); A=pd.read_csv(DATA/'analyst_targets.csv'); sources=pd.read_csv(DATA/'source_documentation.csv')
S=float(M['spot']); K=float(M['strike']); T=float(M['maturity_years']); r=float(M['risk_free_rate']); q=float(M['dividend_yield']); sig=float(M['volatility']); N=int(M['lattice_steps']); mult=int(M['contract_multiplier']); kind=M['option_type']; american=M['exercise_style']=='american'
assert len(A)>=5 and (A.target>0).all(); assert 0<T and 0<sig<3 and 0<=r<=1; assert float(M['market_bid'])<=float(M['market_ask']); assert mult>0

def payoff(s,k=K,kind=kind):
    return np.maximum(np.asarray(s)-k,0) if kind=='call' else np.maximum(k-np.asarray(s),0)
def crr(S,K,T,r,sigma,N,q,kind='call',american=True):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d)
    if not(d<growth<u and 0<p<1): raise ValueError('Invalid risk-neutral probability/no arbitrage')
    stock=S*u**np.arange(N+1)*d**(N-np.arange(N+1)); vals=payoff(stock,K,kind)
    exercise_count=0
    for n in range(N-1,-1,-1):
        vals=np.exp(-r*dt)*(p*vals[1:n+2]+(1-p)*vals[:n+1])
        if american:
            j=np.arange(n+1); intrinsic=payoff(S*u**j*d**(n-j),K,kind)
            exercise_count+=int(np.sum(intrinsic>vals+1e-10));vals=np.maximum(vals,intrinsic)
    return float(vals[0]),float(p),int(exercise_count)
def bs(S,K,T,r,sigma,q,kind='call'):
    if T<=0:return float(payoff(S,K,kind))
    d1=(np.log(S/K)+(r-q+.5*sigma*sigma)*T)/(sigma*np.sqrt(T));d2=d1-sigma*np.sqrt(T)
    if kind=='call':return float(S*np.exp(-q*T)*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2))
    return float(K*np.exp(-r*T)*norm.cdf(-d2)-S*np.exp(-q*T)*norm.cdf(-d1))
# Primary Greeks: analytic European Black-Scholes benchmark; American CRR is priced separately.
d1=(np.log(S/K)+(r-q+.5*sig*sig)*T)/(sig*np.sqrt(T)); d2=d1-sig*np.sqrt(T); pdf=norm.pdf(d1)
delta=np.exp(-q*T)*(norm.cdf(d1) if kind=='call' else norm.cdf(d1)-1)
gamma=np.exp(-q*T)*pdf/(S*sig*np.sqrt(T));vega=S*np.exp(-q*T)*pdf*np.sqrt(T)
theta=-(S*np.exp(-q*T)*pdf*sig)/(2*np.sqrt(T))+(q*S*np.exp(-q*T)*norm.cdf(d1)-r*K*np.exp(-r*T)*norm.cdf(d2) if kind=='call' else -q*S*np.exp(-q*T)*norm.cdf(-d1)+r*K*np.exp(-r*T)*norm.cdf(-d2))
rho=K*T*np.exp(-r*T)*(norm.cdf(d2) if kind=='call' else -norm.cdf(-d2))
greeks={'delta':delta,'gamma':gamma,'vega_per_1vol':vega,'vega_per_1pct_point':vega*.01,'theta_per_year':theta,'theta_per_day':theta/365,'rho_per_1rate':rho,'rho_per_1pct_point':rho*.01}
hS=.10; hsig=.001; hr=.001; ht=1/3650
f=lambda s=S,t=T,vol=sig,rate=r:bs(s,K,t,rate,vol,q,kind)
fd={'delta':(f(s=S+hS)-f(s=S-hS))/(2*hS),'gamma':(f(s=S+hS)-2*f()+f(s=S-hS))/hS**2,'vega_per_1vol':(f(vol=sig+hsig)-f(vol=sig-hsig))/(2*hsig),'theta_per_year':(f(t=T-ht)-f(t=T+ht))/(2*ht),'rho_per_1rate':(f(rate=r+hr)-f(rate=r-hr))/(2*hr)}
fd_table=[dict(greek=k,primary=float(greeks[k]),finite_difference=float(v),bump={'delta':hS,'gamma':hS,'vega_per_1vol':hsig,'theta_per_year':ht,'rho_per_1rate':hr}[k],difference=float(greeks[k]-v)) for k,v in fd.items()]
price,p,exercise_count=crr(S,K,T,r,sig,N,q,kind,american); european,_,_=crr(S,K,T,r,sig,N,q,kind,False)
convergence=[dict(steps=n,price=crr(S,K,T,r,sig,n,q,kind,american)[0]) for n in [25,50,100,200,500]]
call=crr(S,K,T,r,sig,500,q,'call',False)[0];put=crr(S,K,T,r,sig,500,q,'put',False)[0];parity=(call-put)-(S*np.exp(-q*T)-K*np.exp(-r*T))
lo=max(S*np.exp(-q*T)-K*np.exp(-r*T),0) if kind=='call' else max(K*np.exp(-r*T)-S*np.exp(-q*T),0)
hi=S if kind=='call' else K
if american:lo=max(lo,float(payoff(S,K,kind)))
targets=A.target.to_numpy(); labels=['Severe bear','Bear','Base','Bull','Tail']; quantiles=[.05,.25,.5,.75,.95]; weights=np.array([.1,.2,.4,.2,.1]); terminal=np.quantile(targets,quantiles); pay=payoff(terminal);belief=float(np.sum(weights*pay)*np.exp(-r*T))
# Target-based states are 12-month beliefs and are NOT 2-day option payoff forecasts.
dS=2.0; dvol=.01; days=0.25; dr=.001
baseline_bs=f();repriced_stock=f(s=S+dS);repriced_all=f(s=S+dS,vol=sig+dvol,t=T-days/365,rate=r+dr)
delta_only=delta*dS; dg=delta*dS+.5*gamma*dS*dS; combined=dg+vega*dvol+theta*days/365+rho*dr
approx={'stock_move':dS,'delta_only':float(delta_only),'delta_gamma':float(dg),'stock_full_reprice_change':float(repriced_stock-baseline_bs),'delta_only_error':float(delta_only-(repriced_stock-baseline_bs)),'delta_gamma_error':float(dg-(repriced_stock-baseline_bs)),'combined_changes':{'stock':dS,'volatility_decimal':dvol,'elapsed_days':days,'rate_decimal':dr},'combined_approximation':float(combined),'combined_full_reprice_change':float(repriced_all-baseline_bs),'combined_error':float(combined-(repriced_all-baseline_bs))}
checks=[('risk-neutral probability',0<p<1,'CRR p* must lie strictly between zero and one'),('no arbitrage',0<p<1,'d < exp((r-q)dt) < u'),('option price bounds',lo-1e-8<=price<=hi+1e-8,'Intrinsic/discounted lower and upper bounds'),('European parity benchmark',abs(parity)<.05,'European call-put parity residual'),('American exercise comparison',price>=european-1e-8,'American price should not be less than European price'),('lattice convergence',abs(convergence[-1]['price']-convergence[-2]['price'])<.05,'200-to-500 step difference; tolerance $0.05'),('analyst horizon aligned',False,'12-month targets vs 2-day option; cannot treat targets as expiration forecasts'),('bid <= ask',float(M['market_bid'])<=float(M['market_ask']),'Same dated quote'),('rate and volatility scaling',0<sig<3 and 0<=r<=1,'Decimal annual inputs; rate unverified'),('delta range',0<=delta<=1 if kind=='call' else -1<=delta<=0,'European benchmark delta bounds'),('gamma nonnegative',gamma>=0,'Long European benchmark gamma'),('Greek units explicit',True,'Per share; vega/rho per 1.00 and 1 percentage point; theta annual and daily'),('finite difference comparisons',all(abs(z['difference'])<.1 for z in fd_table),'Central finite differences with disclosed bumps'),('approximation vs repricing',True,'Delta-only, delta-gamma and combined shown with errors'),('Greek valuation-date alignment',True,'Same 2026-11-09 model inputs; European benchmark vs American CRR disclosed'),('per-share and contract scaling',True,'100 shares per listed option contract'),('contract-specific implied volatility verified',False,'0.248 is expiration-level chain IV proxy'),('risk-free rate verified',False,'4% assumption requires dated term-matched evidence'),('DCF valuation-date reconciliation',False,'Earlier student DCF $97.20 has not been rolled to 2026-11-09'),('dividend timing',False,'Nov 12 ex-dividend follows Nov 11 expiry; constant yield is a simplifying approximation')]
checks=[dict(name=n,status='PASS' if ok else 'WARNING',explanation=why) for n,ok,why in checks]
xs=np.linspace(S*.88,S*1.12,65);curve=[]
for x in xs:
    a=(np.log(x/K)+(r-q+.5*sig*sig)*T)/(sig*np.sqrt(T));b=a-sig*np.sqrt(T);density=norm.pdf(a)
    curve.append(dict(stock=float(x),delta=float(np.exp(-q*T)*norm.cdf(a)),gamma=float(np.exp(-q*T)*density/(x*sig*np.sqrt(T))),vega_per_1pct_point=float(x*np.exp(-q*T)*density*np.sqrt(T)*.01),theta_per_day=float((-(x*np.exp(-q*T)*density*sig)/(2*np.sqrt(T))+q*x*np.exp(-q*T)*norm.cdf(a)-r*K*np.exp(-r*T)*norm.cdf(b))/365)))
results=dict(model=M,price_crr=price,price_european_crr=european,price_bs_benchmark=baseline_bs,p_star=p,exercise_nodes=exercise_count,greeks=greeks,finite_differences=fd_table,approximation=approx,convergence=convergence,parity_residual=parity,checks=checks,analysts=A.to_dict('records'),analyst_stats={'mean':float(np.mean(targets)),'median':float(np.median(targets)),'min':float(min(targets)),'max':float(max(targets))},scenario_states=[dict(state=l,terminal_price=float(t),weight=float(w),payoff=float(y)) for l,t,w,y in zip(labels,terminal,weights,pay)],analyst_scenario_discounted_payoff=belief,curve=curve,market_mid=(float(M['market_bid'])+float(M['market_ask']))/2)
(DOCS/'aapl_greeks_results.json').write_text(json.dumps(results,indent=2,default=str));print('CRR',price,'BS',baseline_bs,'market mid',results['market_mid']);print(pd.DataFrame(checks).to_string(index=False));print(pd.DataFrame(fd_table).to_string(index=False))


CRR 2.9652690856923365 BS 2.9637847386321994 market mid 3.125
                                         name  status                                                                             explanation
                     risk-neutral probability    PASS                                           CRR p* must lie strictly between zero and one
                                 no arbitrage    PASS                                                                    d < exp((r-q)dt) < u
                          option price bounds    PASS                                             Intrinsic/discounted lower and upper bounds
                    European parity benchmark    PASS                                                       European call-put parity residual
                 American exercise comparison    PASS                                   American price should not be less than European price
                          lattice convergence    PASS                                 

## Decision memo

The Apple November 11 $350 call is an American-style listed equity option. The November 9 stock close was $350.69, while the option bid/ask was $3.00/$3.25 (midpoint $3.125). The CRR price is risk-neutral and sensitive to a **proxy** IV of 24.8%, a **hypothetical** 4% short rate, and a simplified continuous dividend yield. The bid/ask midpoint is a mechanical reference, not an executable guarantee. Analyst targets are generally 12-month forecasts and cannot be treated as two-day terminal outcomes: the displayed analyst scenario payoff is intentionally a horizon-mismatched diagnostic, not an investable fair-value estimate. The old DCF $97.20 was estimated at a different date and cannot be interpreted as a November 9 value without a bridge and diluted-share reconciliation.

**Greeks:** European Black–Scholes local derivatives are reported per share, alongside American CRR valuation. Near-the-money short-dated gamma can be high, causing delta hedge ratios to change quickly. The vega/rho unit conventions and daily theta are explicit. Finite differences validate the European benchmark; they do not independently validate the American CRR Greeks. Stock and multi-factor Taylor estimates are compared with fully repriced European benchmark values. A hedge of 100×delta shares per long call must be adjusted as spot and time change.

**Revision:** Retained original Asteron notebook and JSON in `original/`; replaced current inputs with sourced Apple quotes and analyst reports, added American CRR pricing and European benchmark Greeks, sensitivity figures, finite differences and warning statuses. No protected original tests were changed. **Outstanding evidence:** date-aligned DCF, term-matched rate, contract-specific IV and explicit analyst-report horizon confirmations. No failed test has been modified to force a pass.